# Project 1: House Price Prediction with Machine Learning
**Level:** Beginner  
**Goal:** Predict house prices from area, bedrooms, age, and distance to the city center.

This notebook creates a reproducible illustrative dataset so it runs without downloading files. The results demonstrate the workflow; they are **not real-market estimates**. For a real project, replace the synthetic data with a documented housing dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
n = 600
area = rng.normal(1500, 500, n).clip(450, 4000)
bedrooms = rng.integers(1, 6, n)
age = rng.integers(0, 41, n)
distance = rng.uniform(1, 30, n)
noise = rng.normal(0, 18000, n)
price = 45000 + area*145 + bedrooms*9000 - age*850 - distance*1800 + noise
df = pd.DataFrame({
    "area_sqft": area.round(0).astype(int),
    "bedrooms": bedrooms,
    "age_years": age,
    "distance_km": distance.round(2),
    "price": price.round(0)
})
df.head()

## 1. Explore the dataset

In [ ]:
display(df.describe().round(2))
display(df.isna().sum().rename("missing_values").to_frame())
plt.figure(figsize=(7, 4))
plt.scatter(df["area_sqft"], df["price"], alpha=0.5)
plt.xlabel("Area (sq ft)"); plt.ylabel("Illustrative price")
plt.title("Area vs. price"); plt.tight_layout(); plt.show()

## 2. Split data and train models

In [ ]:
X = df.drop(columns="price")
y = df["price"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
linear = Pipeline([
    ("scale", StandardScaler()),
    ("model", LinearRegression())
])
forest = RandomForestRegressor(n_estimators=250, min_samples_leaf=2,
                               random_state=RANDOM_STATE)
models = {"Linear Regression": linear, "Random Forest": forest}
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    })
metrics = pd.DataFrame(results).sort_values("RMSE")
metrics.round(3)

## 3. Compare results

In [ ]:
ax = metrics.set_index("Model")[["MAE", "RMSE"]].plot(kind="bar", figsize=(7,4))
ax.set_ylabel("Error (price units)"); ax.set_title("Model error comparison")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

best_name = metrics.iloc[0]["Model"]
best_model = models[best_name]
pred = best_model.predict(X_test)
plt.figure(figsize=(6,6))
plt.scatter(y_test, pred, alpha=0.6)
lo, hi = min(y_test.min(), pred.min()), max(y_test.max(), pred.max())
plt.plot([lo, hi], [lo, hi], linestyle="--")
plt.xlabel("Actual illustrative price"); plt.ylabel("Predicted price")
plt.title(f"Actual vs predicted — {best_name}")
plt.tight_layout(); plt.show()

## 4. Predict for a new example

In [ ]:
new_house = pd.DataFrame([{
    "area_sqft": 1800,
    "bedrooms": 3,
    "age_years": 8,
    "distance_km": 7.5
}])
predicted_price = best_model.predict(new_house)[0]
print(f"Selected model: {best_name}")
print(f"Illustrative predicted price: {predicted_price:,.0f} price units")

## 5. Interpretation and limitations
- MAE and RMSE are in the target's price units; lower values indicate smaller errors.
- R² measures how much target variation is explained on this test split; it can be negative on poor predictions.
- Synthetic data is useful for learning but cannot validate performance on actual housing markets.
- Next steps: use a real, licensed dataset; document location and date; inspect outliers; tune hyperparameters; and validate on a holdout set.